# Examine embeds_mean — sanity checks + per-model vs. pooled

Two parts:

1. **Sanity checks** on the aggregation logic in
   `compute_embeds_mean.py`, run against small synthetic arrays (no
   GPU/PRM needed) — per-model mean arithmetic, the pooled mean's
   candidate-weighted definition, save/load round-trip, manifest
   fields, and shape-guard compatibility with
   `_center_and_normalize`. Each cell asserts; a failure here means
   the aggregation logic itself is wrong, before ever touching real
   data.
2. **Analysis** of the real saved means + raw embeddings
   (`results/embeds_mean/level-5/`), asking the question that
   motivated computing them separately per model: **do the 5 LLM
   families actually need their own fixed mean, or would the pooled
   mean serve just as well?**


## Part 1 — sanity checks (synthetic data)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

EMBEDS_DIM = 8


def rng_model(seed, n, dim=EMBEDS_DIM, offset=0.0):
    rng = np.random.default_rng(seed)
    return rng.normal(loc=offset, scale=1.0, size=(n, dim))


### 1a. Per-model mean arithmetic


In [ ]:
pooled = rng_model(seed=0, n=50)
mean = pooled.mean(axis=0, keepdims=True)
assert mean.shape == (1, EMBEDS_DIM)

expected = np.sum(pooled, axis=0, keepdims=True) / pooled.shape[0]
np.testing.assert_allclose(mean, expected)

single = rng_model(seed=2, n=1)
np.testing.assert_allclose(single.mean(axis=0, keepdims=True), single)

print("OK: per-model mean shape, arithmetic, n=1 edge case")


### 1b. Pooled mean is candidate-weighted, NOT the mean of the per-model means

These two definitions only coincide when every model has the same
candidate count — deliberately using unequal counts here so a bug
that silently swapped one definition for the other would be caught.


In [ ]:
model_a = rng_model(seed=10, n=100, offset=0.0)
model_b = rng_model(seed=11, n=10, offset=5.0)

combined = np.concatenate([model_a, model_b], axis=0)
pooled_mean = combined.mean(axis=0, keepdims=True)

mean_of_means = np.mean(
    [model_a.mean(axis=0), model_b.mean(axis=0)], axis=0, keepdims=True,
)

# The larger model (n=100, offset=0) should pull the candidate-
# weighted pooled mean much closer to 0 than the unweighted mean-of-
# means, which treats both models equally despite the 10x count gap.
assert np.abs(pooled_mean).mean() < np.abs(mean_of_means).mean()
assert not np.allclose(pooled_mean, mean_of_means)

# ...but the two DO coincide when counts are equal:
model_c = rng_model(seed=20, n=30, offset=1.0)
model_d = rng_model(seed=21, n=30, offset=-1.0)
combined2 = np.concatenate([model_c, model_d], axis=0)
pooled_mean2 = combined2.mean(axis=0, keepdims=True)
mean_of_means2 = np.mean(
    [model_c.mean(axis=0), model_d.mean(axis=0)], axis=0, keepdims=True,
)
np.testing.assert_allclose(pooled_mean2, mean_of_means2, atol=1e-10)

counts = [50, 30, 10, 90, 20]
models = [rng_model(seed=i, n=n) for i, n in enumerate(counts)]
assert np.concatenate(models, axis=0).shape[0] == sum(counts)

print("OK: pooled mean is concat-then-mean (candidate-weighted), "
      "confirmed to diverge from mean-of-means under unequal counts "
      "and to coincide only when counts are equal")


### 1c. Save/load round-trip


In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmp:
    mean = rng_model(seed=42, n=100).mean(axis=0, keepdims=True)
    out_path = os.path.join(tmp, "embeds_mean--llama_1b--qwen-prm.npy")
    np.save(out_path, mean)

    # Mirrors how generate_mcts_sem.py loads it: np.load(...).flatten()
    loaded = np.load(out_path).flatten()
    np.testing.assert_array_equal(loaded, mean.flatten())
    assert loaded.shape == (EMBEDS_DIM,)

    # All 6 filenames independently loadable, no cross-talk.
    tags = ["llama_1b", "llama_3b", "qwen_3b",
            "qwen_7b_gptq", "qwen_math_1_5b", "pooled"]
    saved = {}
    for i, tag in enumerate(tags):
        m = rng_model(seed=i, n=10).mean(axis=0, keepdims=True)
        np.save(os.path.join(tmp, f"embeds_mean--{tag}--qwen-prm.npy"), m)
        saved[tag] = m
    for tag, m in saved.items():
        reloaded = np.load(os.path.join(tmp, f"embeds_mean--{tag}--qwen-prm.npy"))
        np.testing.assert_array_equal(reloaded, m)

print("OK: .npy round-trips exactly; all 6 files independently loadable")


### 1d. Manifest sidecar — counts and pipeline settings recorded correctly

A `.npy` alone can't say what pipeline or how many candidates
produced it; the manifest must, so a mean found later is never
ambiguous.


In [ ]:
counts = {"llama_1b": 2888, "qwen_3b": 1500}
manifest_check = {
    "models": {
        tag: {
            "num_candidates": n,
            "npy": f"results/embeds_mean/level-5/embeds_mean--{tag}--qwen-prm.npy",
            "raw_npy": f"results/embeds_mean/level-5/raw/{tag}--qwen-prm.npy",
        }
        for tag, n in counts.items()
    },
    "pooled": {"num_candidates": sum(counts.values())},
    "embeds_pipeline": {
        "embeds_strategy": "last", "embeds_scope": "full",
        "embeds_proj": "sparse", "embeds_dim": 512, "prm_embeds_layer": -1,
    },
}

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "embeds_mean--manifest.json")
    with open(path, "w", encoding="utf-8") as fout:
        json.dump(manifest_check, fout)
    reloaded = json.load(open(path, encoding="utf-8"))

for tag, n in counts.items():
    assert reloaded["models"][tag]["num_candidates"] == n
    assert "raw_npy" in reloaded["models"][tag]
assert reloaded["pooled"]["num_candidates"] == sum(counts.values())
assert reloaded["embeds_pipeline"]["embeds_dim"] == 512

print("OK: manifest round-trips counts, npy/raw_npy paths, and "
      "embeds_pipeline settings")


### 1e. Shape-guard compatibility with `_center_and_normalize`

The saved mean must be shape-compatible with the guard in
`core/mcts_sem_search_v02_00_00.py::_center_and_normalize`:
`mean.shape[-1]` must equal the post-projection `embeds_dim` — this
doesn't call the guard directly (it lives in core), just confirms
the shapes that should and shouldn't pass it.


In [ ]:
embeds_dim = 512
pooled_512 = rng_model(seed=0, n=20, dim=embeds_dim)
mean_512 = pooled_512.mean(axis=0, keepdims=True)
assert mean_512.shape[-1] == embeds_dim

# A raw-space (pre-projection) mean would NOT match -- confirms the
# guard's condition actually distinguishes the two cases.
raw_dim, proj_dim = 1536, 512
wrong_mean = rng_model(seed=0, n=5, dim=raw_dim).mean(axis=0, keepdims=True)
stacked = rng_model(seed=1, n=4, dim=proj_dim)
assert wrong_mean.shape[-1] != stacked.shape[-1]

print("OK: post-projection mean shape matches embeds_dim; a "
      "raw-space mean is correctly distinguishable from it")


All sanity checks passed — the aggregation logic itself is sound.
The rest of this notebook analyzes the real data.

## Part 2 — analysis of the real saved means

Loads the means + raw per-candidate embeddings saved by
`compute_embeds_mean.py` (`results/embeds_mean/level-5/`). Three
views, in order of how much they actually answer the "separate vs.
pooled" question:

1. Cosine similarity between the 5 per-model means (+ pooled) — a
   quick first look, but can mislead: two means can look "different"
   while both sitting well inside one model's own spread.
2. **Headline diagnostic: within-model vs. between-model variance
   ratio.** This is the real signal — it normalizes mean-separation
   against how noisy a single model's own embeddings already are.
3. PCA scatter of raw embeddings with means overlaid, for a visual
   sanity check of (2).


In [ ]:
MEAN_DIR = "results/embeds_mean/level-5"

with open(f"{MEAN_DIR}/embeds_mean--manifest.json", encoding="utf-8") as fin:
    manifest = json.load(fin)

print(f"created_at: {manifest['created_at']}")
print(f"prm: {manifest['prm']}")
print(f"embeds_pipeline: {manifest['embeds_pipeline']}")
print()
for tag, info in manifest["models"].items():
    print(f"{tag:16s} n={info['num_candidates']:5d}  {info['npy']}")
print(f"{'pooled':16s} n={manifest['pooled']['num_candidates']:5d}  {manifest['pooled']['npy']}")


In [ ]:
# Load means (each (1, embeds_dim)) and raw per-candidate arrays
# (each (n_candidates, embeds_dim)).
model_tags = list(manifest["models"].keys())

means = {}
raw = {}
for tag in model_tags:
    means[tag] = np.load(manifest["models"][tag]["npy"]).flatten()
    raw[tag] = np.load(manifest["models"][tag]["raw_npy"])
means["pooled"] = np.load(manifest["pooled"]["npy"]).flatten()

real_embeds_dim = means[model_tags[0]].shape[0]
print(f"embeds_dim = {real_embeds_dim}")
print(f"models: {model_tags}")


### 2a. Cosine similarity between means


In [ ]:
all_tags = model_tags + ["pooled"]
mean_matrix = np.stack([means[t] for t in all_tags])  # (n_models+1, dim)

norms = np.linalg.norm(mean_matrix, axis=1, keepdims=True)
unit_means = np.divide(
    mean_matrix, norms, out=np.zeros_like(mean_matrix), where=norms > 0,
)
cosine_sim = unit_means @ unit_means.T

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cosine_sim, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(all_tags)))
ax.set_yticks(range(len(all_tags)))
ax.set_xticklabels(all_tags, rotation=45, ha="right")
ax.set_yticklabels(all_tags)
for i in range(len(all_tags)):
    for j in range(len(all_tags)):
        ax.text(j, i, f"{cosine_sim[i, j]:.2f}", ha="center", va="center",
                 color="black", fontsize=9)
ax.set_title("Cosine similarity between means")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()


In [ ]:
# Distance of each per-model mean from the pooled mean, as a fraction
# of the pooled mean's own norm -- a scale-normalized "how far is
# this model's mean from the population mean" number.
pooled_norm = np.linalg.norm(means["pooled"])
print(f"||pooled mean|| = {pooled_norm:.4f}\n")
for tag in model_tags:
    dist = np.linalg.norm(means[tag] - means["pooled"])
    print(f"{tag:16s} ||mean - pooled|| = {dist:.4f}  "
          f"({dist / pooled_norm:.1%} of pooled norm)")


### 2b. Headline diagnostic: within-model vs. between-model variance

The real question isn't "are the means different" (they always will
be, numerically) but "are they different **relative to how much a
single model's own embeddings already vary**?" If between-model
separation is small next to within-model spread, per-model means are
mostly fitting noise — the pooled mean should do just as well in
practice. If between-model separation is large relative to
within-model spread, the models really do occupy distinguishable
regions and separate means are earning their keep.

- **Within-model variance**: mean pairwise squared distance between
  raw candidate embeddings, computed *inside* each model, then
  averaged across models.
- **Between-model variance**: mean pairwise squared distance between
  the 5 per-model *means*.
- **Ratio = between / within.** Ratio << 1 -> means are well inside
  the noise floor (pooled mean is fine). Ratio >~ 1 -> real
  separation (per-model means matter).


In [ ]:
def mean_pairwise_sq_dist(X, max_samples=2000, seed=0):
    """Mean pairwise squared Euclidean distance, subsampled for
    speed on the raw (thousands x dim) arrays."""
    n = X.shape[0]
    if n > max_samples:
        rng = np.random.default_rng(seed)
        idx = rng.choice(n, size=max_samples, replace=False)
        X = X[idx]
    sq_norms = np.sum(X ** 2, axis=1)
    sq_dists = sq_norms[:, None] + sq_norms[None, :] - 2 * (X @ X.T)
    iu = np.triu_indices(X.shape[0], k=1)
    return float(np.mean(np.clip(sq_dists[iu], a_min=0, a_max=None)))


within_model_var = {
    tag: mean_pairwise_sq_dist(raw[tag]) for tag in model_tags
}
avg_within = float(np.mean(list(within_model_var.values())))

mean_stack = np.stack([means[t] for t in model_tags])
between_model_var = mean_pairwise_sq_dist(mean_stack)

ratio = between_model_var / avg_within

print("within-model variance (mean pairwise sq. dist, per model):")
for tag, v in within_model_var.items():
    print(f"  {tag:16s} {v:.4f}")
print(f"  average across models: {avg_within:.4f}\n")

print(f"between-model variance (pairwise sq. dist between the 5 means): "
      f"{between_model_var:.4f}\n")

print(f"RATIO (between / within) = {ratio:.4f}")
if ratio < 0.05:
    verdict = "well inside the noise floor -- pooled mean likely suffices"
elif ratio < 0.2:
    verdict = "small but non-trivial -- worth watching, not conclusive"
else:
    verdict = "real separation -- per-model means likely matter"
print(f"-> {verdict}")


### 2c. PCA scatter — visual check

Projects a sample of each model's raw embeddings onto the top-2
principal components (fit on the pooled sample), with each model's
mean overlaid as a large marker. If the diagnostic above says the
ratio is small, expect heavy overlap between the clouds below; if
large, expect visibly separated clusters.


In [ ]:
from numpy.linalg import svd

SAMPLE_PER_MODEL = 300

rng = np.random.default_rng(0)
samples = {}
for tag in model_tags:
    X = raw[tag]
    n = min(SAMPLE_PER_MODEL, X.shape[0])
    idx = rng.choice(X.shape[0], size=n, replace=False)
    samples[tag] = X[idx]

pooled_sample = np.concatenate(list(samples.values()), axis=0)
pooled_sample_mean = pooled_sample.mean(axis=0, keepdims=True)
centered = pooled_sample - pooled_sample_mean

# PCA via SVD (no sklearn dependency needed for just 2 components).
U, S, Vt = svd(centered, full_matrices=False)
components = Vt[:2]  # (2, embeds_dim)

fig, ax = plt.subplots(figsize=(8, 7))
colors = plt.cm.tab10(np.linspace(0, 1, len(model_tags)))
for tag, color in zip(model_tags, colors):
    proj = (samples[tag] - pooled_sample_mean) @ components.T
    ax.scatter(proj[:, 0], proj[:, 1], s=8, alpha=0.35, color=color,
               label=f"{tag} (candidates)")
    mean_proj = (means[tag] - pooled_sample_mean.flatten()) @ components.T
    ax.scatter(*mean_proj, s=250, marker="X", color=color,
               edgecolor="black", linewidth=1.2, zorder=5)

pooled_mean_proj = (
    (means["pooled"] - pooled_sample_mean.flatten()) @ components.T
)
ax.scatter(*pooled_mean_proj, s=350, marker="*", color="black",
           label="pooled mean", zorder=6)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Per-model candidate embeddings + means (top-2 PCA)")
ax.legend(loc="best", fontsize=8, markerscale=1.5)
plt.tight_layout()
plt.show()


## Reading this together

- If the variance-ratio in 2b is small AND the PCA clouds in 2c
  mostly overlap: the pooled mean is doing the same job as 5 separate
  means would, at a fraction of the bookkeeping (one
  `embeds_mean_dir` instead of 5 across configs).
- If the ratio is larger AND the clouds show visible separation: the
  per-model means are earning their keep, and search runs should set
  `embeds_mean_dir` to the matching per-model `.npy`, not the pooled
  one.
- Either way, the per-model `.npy`s are already saved
  (`results/embeds_mean/level-5/embeds_mean--<tag>--qwen-prm.npy`) —
  this notebook only decides which one a given sem-mcts config should
  point `search.embeds_mean_dir` at.
